# Early Sepsis Warning from ICU Time Series

This notebook audits the PhysioNet 2019 training data and runs the reproducible pipeline in `src/run_pipeline.py`. It is an educational research prototype, not a clinical decision tool. The challenge label begins six hours before clinical onset and remains positive afterward, so patient-level alert timing matters alongside row-level metrics.

## 1. Setup and data access

Download the open-access training archive from [PhysioNet](https://physionet.org/content/challenge-2019/1.0.0/), extract it, and put the patient `.psv` files under `data/raw/`. The loader searches subfolders recursively.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT is None and (Path.cwd() / 'outputs' / 'sepsis_early_warning' / 'src').is_dir():
    PROJECT = Path.cwd() / 'outputs' / 'sepsis_early_warning'
if PROJECT is None:
    raise FileNotFoundError('Could not locate the sepsis project folder; open this notebook from its project directory.')
sys.path.insert(0, str(PROJECT))
from src.run_pipeline import read_patients

DATA_DIR = PROJECT / 'data' / 'raw'
data = read_patients(DATA_DIR)
data.shape, data['patient_id'].nunique()

## 2. Cohort and label audit

The unit of splitting is the patient, not the hourly row. The positive label is rare at the patient level and the hourly level; show both so the evaluation setup is transparent.

In [ ]:
patient_summary = data.groupby('patient_id').agg(
    hours=('SepsisLabel', 'size'), has_sepsis=('SepsisLabel', 'max'),
    positive_hours=('SepsisLabel', 'sum')
)
display(patient_summary.describe().T)
print(f"Patients: {len(patient_summary):,}; positive patients: {patient_summary.has_sepsis.sum():,}")
print(f"Hourly positive fraction: {data.SepsisLabel.mean():.3%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
patient_summary.hours.hist(ax=axes[0], bins=40)
axes[0].set(title='ICU hours per patient', xlabel='Hours', ylabel='Patients')
patient_summary.positive_hours.hist(ax=axes[1], bins=30)
axes[1].set(title='Positive-label hours per patient', xlabel='Positive hours', ylabel='Patients')
plt.tight_layout()

## 3. Missingness and temporal structure

Clinical measurements are sampled irregularly. Missingness indicators and causal trailing summaries can encode both what was measured and how often. No future rows are used to build features.

In [ ]:
measurement_columns = [c for c in data.select_dtypes('number').columns if c not in {'SepsisLabel', 'hour_index'}]
missing = data[measurement_columns].isna().mean().sort_values(ascending=False)
display(missing.head(15).rename('missing_fraction').to_frame())
missing.head(20).sort_values().plot.barh(figsize=(8, 6), title='Most frequently missing measurements')
plt.xlabel('Fraction missing')
plt.tight_layout()

## 4. Modeling and held-out evaluation

Run from the project root with `python -m src.run_pipeline --data-dir data/raw --output-dir outputs`, then `python -m src.analyze_alerts --output-dir outputs`. The pipeline reserves a patient-level test split, selects a threshold on validation patients using the official challenge utility, compares a class-weighted logistic model and histogram gradient boosting against a prevalence baseline, and writes metrics and predictions as CSV/JSON. Interpret row-level metrics cautiously alongside normalized patient-level utility.

In [ ]:
metrics = pd.read_csv(PROJECT / 'outputs' / 'model_comparison.csv')
patient_alerts = pd.read_csv(PROJECT / 'outputs' / 'patient_alert_analysis.csv')
display(metrics[['model', 'average_precision', 'roc_auc', 'recall_sensitivity', 'precision', 'normalized_challenge_utility']])
display(patient_alerts.groupby('has_sepsis').agg(patients=('patient_id', 'nunique')))